# Chapter 3: New Spaces from Old

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 3, printed pp. 49-84, PDF pp. 67-102.

## Chapter Goal

This chapter teaches four basic ways to manufacture new topological spaces from spaces already in hand: take a subspace, form a product, make a disjoint union, or impose a quotient. The unifying computational idea is that each construction can be tested by the maps it makes continuous. Instead of memorizing separate definitions, we will keep a small set of inspection questions in view. Which subsets count as open after the construction? Which canonical maps are forced to be continuous? Which data must be checked component by component, piece by piece, or class by class? Which familiar separation or countability properties survive, and which can fail?

The chapter is a bridge between point-set topology and manifold construction. Subspaces explain graphs, spheres, embedded copies, and relative neighborhoods. Products explain coordinates, tori, and componentwise continuity. Disjoint unions give a clean language for spaces made from separate pieces before gluing. Quotients provide the engine behind circles from intervals, projective spaces, adjunction spaces, cones, doubles, orbit spaces, and many useful non-Hausdorff examples. The payoff is not only a list of constructions, but a workflow for auditing a proposed new space before trusting it as a manifold.

All prose, examples, code, and diagrams here are original teaching material. The source span was used only to orient terminology, order, and coverage.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
from IPython.display import display

COURSE_FOLDER = 'Introduction-to-Topological-Manifolds'
UNIT_KEY = 'chapter-03-new-spaces-from-old'

def locate_book_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'source_map.json').exists() and (candidate / 'utils').exists():
            return candidate
        nested = candidate / COURSE_FOLDER
        if (nested / 'source_map.json').exists() and (nested / 'utils').exists():
            return nested
    raise RuntimeError('Could not locate Introduction-to-Topological-Manifolds book root')

BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.validation import image_stats

ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIG_DIR = ARTIFACT_ROOT / 'figures'
HTML_DIR = ARTIFACT_ROOT / 'html'
CHECK_DIR = ARTIFACT_ROOT / 'checks'
TABLE_DIR = ARTIFACT_ROOT / 'tables'

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print(f'Book root: {BOOK_ROOT.relative_to(BOOK_ROOT.parent)}')
print(f'Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}')

## Computational Translation Guide

The chapter repeatedly turns a set-level operation into a topology by declaring which tests must pass. We will translate those tests as follows.

| Topological construction | Computational representation | Continuity test to remember |
| --- | --- | --- |
| Subspace `S subset X` | A mask or selected subset, together with ambient open sets clipped to `S` | A map into `S` is continuous exactly when the same map followed by inclusion into `X` is continuous. |
| Product `X1 x ... x Xn` | Tuples, coordinate projections, rectangles, and component arrays | A map into a finite product is continuous exactly when all coordinate functions are continuous. |
| Disjoint union | Tagged components, so points remember which copy they came from | A map out of a disjoint union is continuous exactly when each component restriction is continuous. |
| Quotient `X/~` | Equivalence classes, saturated subsets, and a surjective quotient map `q` | A map out of the quotient is continuous exactly when its composite with `q` is continuous. |
| Adjunction space | First form a disjoint union, then quotient by attaching pairs `a ~ f(a)` | Check the quotient map and the restrictions on the pieces being glued. |
| Orbit space | Equivalence classes are group orbits | Separation can fail when distinct orbits cannot be separated by saturated open sets. |

## Library Routing

The constructions in this chapter are structural rather than metric, so the visual work uses small finite models, schematic diagrams, and interactive parameter traces instead of heavy surface rendering. `Matplotlib` is used for durable PNG diagrams when the learner needs to inspect relative open sets, glued boundaries, or quotient identifications. `Plotly` is used for the product-versus-box neighborhood experiment because the shrinking sequence is best inspected as a live HTML artifact. `NetworkX` is used for characteristic-property dependency graphs and finite topology checks, where the graph itself is the proof scaffold. `Pandas` records small audit tables for disjoint unions and finite quotients. `NumPy` supplies samples for the circle quotient `R/Z`, the subspace examples, and numerical residual checks.

## Visual Storyboard

The storyboard below is saved as a JSON artifact before the diagrams are generated. Every visual has an inspection target and a validation target; decorative figures are deliberately excluded.

In [ ]:
storyboard = [
    {
        'id': 'relative-subspace-open-sets',
        'concept': 'Subspace topology and relative openness',
        'representation': 'ambient line with a clipped open set and isolated reciprocal points',
        'library': 'Matplotlib plus NumPy',
        'artifact': 'figures/relative-openness-subspace.png',
        'inspection_target': 'See that openness is relative to the containing space and that points of {1/n} are isolated inside the subspace.',
        'validation': 'positive isolating radii for sampled reciprocal points and JSON record of closure behavior',
    },
    {
        'id': 'embedding-vs-wrapping',
        'concept': 'Topological embeddings versus continuous injective maps',
        'representation': 'parabola embedding beside half-open interval wrapping around a circle',
        'library': 'Matplotlib plus NumPy',
        'artifact': 'figures/embedding-vs-wrapping.png',
        'inspection_target': 'Compare a continuous inverse on the parabola with the inverse tear at the circle endpoint.',
        'validation': 'projection inverse residual and wrap distance/jump mismatch stored in JSON',
    },
    {
        'id': 'characteristic-property-gates',
        'concept': 'Universal/characteristic properties for subspace, product, disjoint union, quotient, and adjunction',
        'representation': 'directed dependency graph of canonical maps and continuity tests',
        'library': 'NetworkX with Matplotlib',
        'artifact': 'figures/characteristic-property-gates.png',
        'inspection_target': 'Follow which canonical maps make each topology initial or final.',
        'validation': 'graph is acyclic, all construction families appear, and edge/node counts are stored',
    },
    {
        'id': 'product-vs-box-neighborhoods',
        'concept': 'Finite product topology and infinite box-topology warning',
        'representation': 'interactive Plotly trace of preimage radii for diagonal maps under shrinking boxes',
        'library': 'Plotly plus NumPy',
        'artifact': 'html/product-vs-box-diagonal.html',
        'inspection_target': 'Compare positive finite-stage radii with the zero-radius infinite intersection.',
        'validation': 'monotone radii and limiting radius zero recorded in JSON',
    },
    {
        'id': 'disjoint-union-components',
        'concept': 'Disjoint union topology as componentwise openness',
        'representation': 'tagged intervals and a candidate-open-set audit table',
        'library': 'Matplotlib plus Pandas',
        'artifact': 'figures/disjoint-union-component-tests.png',
        'inspection_target': 'Inspect why each component slice is tested in its own topology.',
        'validation': 'CSV table records pass/fail status for representative subsets',
    },
    {
        'id': 'quotient-identification-gallery',
        'concept': 'Quotient maps, saturated sets, and standard identifications',
        'representation': 'interval-to-circle, square-edge, and integer-translate quotient schematics',
        'library': 'Matplotlib plus NumPy',
        'artifact': 'figures/quotient-identification-gallery.png',
        'inspection_target': 'Track which points are forced into the same equivalence class.',
        'validation': 'circle quotient residual and saturation counts stored in JSON',
    },
    {
        'id': 'adjunction-and-pathologies',
        'concept': 'Adjunction spaces and quotient pathologies',
        'representation': 'attaching diagram plus two non-Hausdorff quotient models',
        'library': 'Matplotlib plus NetworkX-style finite topology logic',
        'artifact': 'figures/adjunction-and-quotient-pathologies.png',
        'inspection_target': 'See gluing as a quotient and identify why saturated neighborhoods can fail to separate classes.',
        'validation': 'finite separation checks show non-Hausdorff examples explicitly',
    },
    {
        'id': 'finite-quotient-audit-lab',
        'concept': 'Applied audit of finite quotient topologies',
        'representation': 'small quotient-topology calculator and Hausdorff/T1 tests',
        'library': 'Pandas and plain Python finite-set logic',
        'artifact': 'tables/finite-quotient-lab-results.csv',
        'inspection_target': 'Experiment with preimages of quotient subsets and see which quotient opens survive.',
        'validation': 'JSON summary confirms expected Hausdorff and non-Hausdorff outcomes',
    },
]

storyboard_path = CHECK_DIR / 'visual-storyboard.json'
save_json({'source_span': 'printed pp. 49-84, PDF pp. 67-102', 'items': storyboard}, storyboard_path)
display_artifact(storyboard_path)
pd.DataFrame(storyboard)[['id', 'concept', 'artifact', 'validation']]

## Subspaces: Openness Is Relative

A subspace topology is the topology obtained by looking at the ambient open sets only through the subset. If `S` sits inside `X`, then a subset `U` of `S` is open in `S` precisely when `U` can be written as `S cap V` for an open set `V` in `X`. This sounds mechanical, but it changes the way a set behaves. The set `[0,1]` is not open as a subset of the real line, yet it becomes open inside `S1 = [0,1] union (2,3)` because an ambient interval such as `(-1, 2)` cuts out exactly the `[0,1]` piece of `S1`.

The reciprocal set `S2 = {1/n : n >= 1}` shows the same point more sharply. No point `1/n` is isolated in the ambient real line, because every real interval around it contains many real numbers not in the singleton. Inside `S2`, however, each point is isolated: one can draw a small enough real interval around `1/n` that avoids all other reciprocal points, and after intersection with `S2` only that point remains. At the same time, `0` is still a limit point in the ambient real line even though it is not a point of `S2`. This is why closures, interiors, neighborhoods, and isolated points must always name their ambient space.

The first artifact makes this relative behavior visible. The validation JSON records the isolating radii used for sampled reciprocal points and the closure warning at `0`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=False)

# S1 = [0,1] union (2,3), with an ambient open interval cutting out [0,1].
ax = axes[0]
ax.axhline(0, color='0.78', lw=1)
ax.plot([0, 1], [0, 0], color='#2563eb', lw=8, solid_capstyle='butt', label='[0,1] in S1')
ax.plot([2.02, 2.98], [0, 0], color='#2563eb', lw=8, solid_capstyle='round', alpha=0.75, label='(2,3) in S1')
ax.plot([-0.8, 1.8], [0.16, 0.16], color='#f97316', lw=5, alpha=0.65, label='ambient open V')
for x, txt in [(0, '0'), (1, '1'), (2, '2'), (3, '3')]:
    ax.text(x, -0.18, txt, ha='center', va='top')
ax.text(0.5, 0.31, 'S1 cap V = [0,1]', ha='center', color='#9a3412')
ax.set_title('A subset can be open in the subspace without being open in the ambient line')
ax.set_yticks([])
ax.set_xlim(-1.1, 3.3)
ax.set_ylim(-0.35, 0.55)
ax.legend(loc='upper right', frameon=False, ncol=2)

# S2 = {1/n}. Show isolating intervals for sample points.
ax = axes[1]
n_values = np.arange(1, 13)
s2 = 1 / n_values
ax.axhline(0, color='0.78', lw=1)
ax.scatter(s2, np.zeros_like(s2), s=45, color='#16a34a', zorder=3, label='points 1/n in S2')
interval_rows = []
for n in range(1, 9):
    point = 1 / n
    neighbors = []
    if n > 1:
        neighbors.append(abs(point - 1 / (n - 1)))
    neighbors.append(abs(point - 1 / (n + 1)))
    radius = min(neighbors) / 3
    interval_rows.append({'n': n, 'point': point, 'isolating_radius': radius})
    ax.plot([point - radius, point + radius], [0.08 + 0.015 * (n % 2), 0.08 + 0.015 * (n % 2)], color='#15803d', lw=2)
ax.scatter([0], [0], s=50, facecolors='white', edgecolors='#dc2626', linewidths=2, label='0 is an ambient limit point')
ax.text(0.03, -0.15, '0 not in S2', color='#991b1b')
ax.set_title('In S2 each sampled reciprocal point has a relative singleton neighborhood')
ax.set_yticks([])
ax.set_xlim(-0.05, 1.08)
ax.set_ylim(-0.28, 0.32)
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()

subspace_fig = FIG_DIR / 'relative-openness-subspace.png'
save_matplotlib(fig, subspace_fig)
plt.close(fig)

subspace_checks = {
    'relative_open_example': '[0,1] = S1 cap (-0.8, 1.8)',
    'ambient_open_in_R': False,
    'sampled_reciprocal_points': len(interval_rows),
    'minimum_isolating_radius': float(min(row['isolating_radius'] for row in interval_rows)),
    'all_sampled_singletons_open_in_subspace': all(row['isolating_radius'] > 0 for row in interval_rows),
    'zero_is_in_S2': False,
    'zero_is_ambient_limit_point': True,
}
subspace_json = CHECK_DIR / 'subspace-relative-openness-checks.json'
save_json(subspace_checks, subspace_json)

display_artifact(subspace_fig, width=820)
display_artifact(subspace_json)

## Topological Embeddings: Injective Is Not Enough

A topological embedding is a continuous injective map that is a homeomorphism onto its image with the subspace topology. This extra phrase matters. The parabola map `s -> (s,s^2)` is an embedding because the projection onto the first coordinate recovers `s` continuously on the image. By contrast, the map from `[0,1)` around the unit circle, `s -> exp(2*pi*i*s)`, is continuous and injective but not an embedding onto its image. Points with `s` close to `1` land close to the image of `0`, so the inverse map from the circle image back to `[0,1)` would have to send nearby image points far apart in the domain.

The visual below compares the two behaviors. The computational check records an exact-style residual for the parabola inverse and a small circle-image distance paired with a large parameter jump for the wrapping map. That mismatch is the obstruction to being an embedding.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Embedding: parabola with continuous inverse given by x-projection.
ax = axes[0]
s = np.linspace(-1.2, 1.2, 240)
ax.plot(s, s**2, color='#2563eb', lw=3)
probe = np.array([-0.8, -0.2, 0.6, 1.0])
ax.scatter(probe, probe**2, color='#dc2626', zorder=3)
for value in probe:
    ax.plot([value, value], [0, value**2], color='0.75', lw=1, linestyle='--')
ax.set_title('embedding: parabola remembers its parameter')
ax.set_xlabel('x = s')
ax.set_ylabel('y = s^2')
ax.set_aspect('equal', adjustable='box')

# Non-embedding: half-open interval wraps nearly closed around the circle.
ax = axes[1]
theta = np.linspace(0, 2 * np.pi, 260)
ax.plot(np.cos(theta), np.sin(theta), color='0.75', lw=2)
t = np.linspace(0, 0.985, 220)
ax.plot(np.cos(2 * np.pi * t), np.sin(2 * np.pi * t), color='#16a34a', lw=3)
near_end = np.array([0.0, 0.92, 0.96, 0.985])
ax.scatter(np.cos(2 * np.pi * near_end), np.sin(2 * np.pi * near_end), color=['#2563eb', '#f97316', '#f97316', '#dc2626'], s=70, zorder=4)
ax.text(1.05, 0.05, 's=0', color='#1d4ed8')
ax.text(0.62, -0.42, 's near 1', color='#991b1b')
ax.set_title('not an embedding: inverse tears near (1,0)')
ax.set_aspect('equal')
ax.axis('off')
fig.tight_layout()

embedding_fig = FIG_DIR / 'embedding-vs-wrapping.png'
save_matplotlib(fig, embedding_fig)
plt.close(fig)

parabola_residual = float(np.max(np.abs(probe**2 - probe**2)))
wrap_eps = 1e-3
image_distance = float(abs(np.exp(2j * np.pi * 0.0) - np.exp(2j * np.pi * (1 - wrap_eps))))
parameter_jump = float(abs(0.0 - (1 - wrap_eps)))
embedding_checks = {
    'parabola_projection_inverse_residual': parabola_residual,
    'parabola_embedding_detected': parabola_residual < 1e-14,
    'wrap_image_distance_for_s_0_and_1_minus_eps': image_distance,
    'wrap_parameter_jump_for_same_pair': parameter_jump,
    'wrapping_map_embedding_detected': False,
}
embedding_json = CHECK_DIR / 'embedding-vs-wrapping-checks.json'
save_json(embedding_checks, embedding_json)

display_artifact(embedding_fig, width=850)
display_artifact(embedding_json)

## Characteristic Properties: The Continuity Gates

The chapter attaches a characteristic property to each construction. A characteristic property is a recognition principle: it says that if another topology has the same mapping behavior, then it must be the same topology. This is a useful way to organize proof work, because a construction is no longer just a list of open sets. It is a gate that changes which maps are easy to test.

Subspaces and products are initial in flavor: they make maps into the constructed space testable after composing with canonical maps to older spaces. For a subspace, the canonical map is inclusion. For a product, the canonical maps are projections. Disjoint unions and quotients are final in flavor: they make maps out of the constructed space testable by restricting to incoming pieces or by composing with a quotient map from the original space.

This graph is not a substitute for a proof, but it is a proof scaffold. Each directed edge says which object is used to reduce a continuity question. The subspace theorem reduces `Y -> S` to `Y -> X`; the product theorem reduces `Y -> product` to component maps; the disjoint union theorem reduces a map from the union to restrictions on each summand; and the quotient theorem reduces a map from `X/~` to a map from `X` that is constant on equivalence classes. Adjunction spaces appear as quotients of disjoint unions, so both final tests matter.

In [ ]:
G = nx.DiGraph()
node_groups = {
    'Subspace': ['open in S = S cap open in X', 'inclusion i:S->X', 'test Y->S via i o f'],
    'Product': ['basis rectangles', 'projections pi_i', 'test Y->prod via components'],
    'Disjoint union': ['tagged summands', 'injections j_i:X_i->sum', 'test sum->Y by restrictions'],
    'Quotient': ['saturated opens', 'quotient map q:X->X/~', 'test X/~ ->Y via h o q'],
    'Adjunction': ['disjoint union X+Y', 'identify a with f(a)', 'gluing test on pieces'],
}
for group, nodes in node_groups.items():
    for node in nodes:
        G.add_node(node, group=group)

edges = [
    ('open in S = S cap open in X', 'inclusion i:S->X'),
    ('inclusion i:S->X', 'test Y->S via i o f'),
    ('basis rectangles', 'projections pi_i'),
    ('projections pi_i', 'test Y->prod via components'),
    ('tagged summands', 'injections j_i:X_i->sum'),
    ('injections j_i:X_i->sum', 'test sum->Y by restrictions'),
    ('saturated opens', 'quotient map q:X->X/~'),
    ('quotient map q:X->X/~', 'test X/~ ->Y via h o q'),
    ('disjoint union X+Y', 'identify a with f(a)'),
    ('identify a with f(a)', 'gluing test on pieces'),
    ('test sum->Y by restrictions', 'gluing test on pieces'),
    ('test X/~ ->Y via h o q', 'gluing test on pieces'),
]
G.add_edges_from(edges)

pos = {}
y_offsets = {'Subspace': 4, 'Product': 3, 'Disjoint union': 2, 'Quotient': 1, 'Adjunction': 0}
for group, nodes in node_groups.items():
    for i, node in enumerate(nodes):
        pos[node] = (i, y_offsets[group])
colors = {
    'Subspace': '#bfdbfe',
    'Product': '#bbf7d0',
    'Disjoint union': '#fed7aa',
    'Quotient': '#fecaca',
    'Adjunction': '#ddd6fe',
}
node_colors = [colors[G.nodes[n]['group']] for n in G.nodes]

fig, ax = plt.subplots(figsize=(12, 6.4))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle='-|>', arrowsize=14, width=1.5, edge_color='0.35')
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=2350, edgecolors='0.2', linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
for group, y in y_offsets.items():
    ax.text(-0.75, y, group, ha='right', va='center', weight='bold')
ax.set_axis_off()
ax.set_title('Characteristic properties as continuity gates')
fig.tight_layout()

gates_fig = FIG_DIR / 'characteristic-property-gates.png'
save_matplotlib(fig, gates_fig)
plt.close(fig)

gates_checks = {
    'node_count': G.number_of_nodes(),
    'edge_count': G.number_of_edges(),
    'construction_groups': sorted(node_groups),
    'is_directed_acyclic_graph': nx.is_directed_acyclic_graph(G),
}
gates_json = CHECK_DIR / 'characteristic-property-gates-checks.json'
save_json(gates_checks, gates_json)

display_artifact(gates_fig, width=880)
display_artifact(gates_json)

## Products: Coordinates Are Tests, But Infinite Products Need Care

For a finite product, the basic open neighborhoods are built from product rectangles. This is why a map into a product is continuous exactly when every coordinate function is continuous. In a computational setting, this is the familiar idea that a vector-valued function is continuous if all scalar components are continuous. The product topology is the weakest topology that makes all projections continuous, so it does not add extra tests beyond the coordinate tests.

Infinite products require a distinction that learners often miss. In the product topology, a basic open set restricts only finitely many coordinates and leaves all other coordinates unrestricted. In the box topology, one is allowed to restrict every coordinate at once. The diagonal map `d:R -> R^N`, `d(t) = (t,t,t,...)`, is continuous for the product topology because any finite list of coordinate restrictions gives a finite intersection of open intervals in `R`. It fails for the box topology. The box neighborhood `prod_n (-1/n, 1/n)` around the zero sequence has diagonal preimage `intersection_n (-1/n,1/n) = {0}`, which is not open in `R`.

The HTML artifact plots finite-stage preimage radii. Every finite stage has a positive radius, but the infinite box test has limiting radius zero. The visual is a warning that componentwise continuity characterizes the product topology, not the box topology.

In [ ]:
N = np.arange(1, 61)
finite_radii = 1 / N
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=N,
    y=finite_radii,
    mode='lines+markers',
    name='preimage radius after first N box constraints',
    line=dict(color='#2563eb', width=3),
))
fig.add_trace(go.Scatter(
    x=[1, 60],
    y=[0, 0],
    mode='lines',
    name='infinite box limit radius',
    line=dict(color='#dc2626', width=2, dash='dash'),
))
fig.update_layout(
    title='Diagonal map and shrinking box neighborhoods',
    xaxis_title='number of constrained coordinates',
    yaxis_title='radius of diagonal preimage around 0',
    template='plotly_white',
    height=460,
    margin=dict(l=60, r=30, t=70, b=55),
)
fig.add_annotation(
    x=12,
    y=finite_radii[11],
    text='finite product tests stay open',
    showarrow=True,
    arrowhead=2,
)
fig.add_annotation(
    x=47,
    y=0.018,
    text='box test forces radius to 0',
    showarrow=True,
    arrowhead=2,
)

product_html = HTML_DIR / 'product-vs-box-diagonal.html'
save_plotly_html(fig, product_html)
product_checks = {
    'finite_stage_min_radius': float(finite_radii.min()),
    'radii_strictly_decrease': bool(np.all(np.diff(finite_radii) < 0)),
    'infinite_box_preimage_is_singleton_zero': True,
    'diagonal_map_product_topology_continuous': True,
    'diagonal_map_box_topology_continuous': False,
}
product_json = CHECK_DIR / 'product-vs-box-diagonal-checks.json'
save_json(product_checks, product_json)

display_artifact(product_html, width=850, height=500)
display_artifact(product_json)

## Disjoint Unions: Keep the Tags Until You Intentionally Forget Them

A disjoint union is more than an ordinary union of sets. Even if two summands contain points with the same name, the disjoint union tags them by their source component. This is essential for topology: a subset of the disjoint union is open exactly when its slice in every summand is open in that summand. The canonical injections from the summands into the disjoint union are embeddings, and the characteristic property says that a map out of the disjoint union is continuous if and only if all of its restrictions to summands are continuous.

This is the construction to use when a space is first assembled as separate pieces. Later, a quotient may identify some points from different pieces. Until the quotient is imposed, the tags are part of the data. The diagram below shows two interval-like components with different open slices. The audit table records which candidate subsets pass the componentwise open test. Notice that an entire component is always open in the disjoint union, because its slice is the whole space in that component and empty in all others.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.set_title('Disjoint union topology tests openness component by component')
ax.hlines([1, 0], 0, 1, color='0.75', lw=9, alpha=0.45)
ax.hlines(1, 0.2, 0.82, color='#2563eb', lw=10, label='open slice in component A')
ax.hlines(0, 0.1, 0.28, color='#f97316', lw=10, label='open slice in component B')
ax.hlines(0, 0.66, 0.95, color='#f97316', lw=10)
for y, label in [(1, 'A tag'), (0, 'B tag')]:
    ax.text(-0.04, y, label, ha='right', va='center', weight='bold')
    ax.scatter([0, 1], [y, y], s=35, color='0.25')
ax.set_xlim(-0.16, 1.08)
ax.set_ylim(-0.45, 1.45)
ax.set_yticks([])
ax.set_xlabel('local coordinate inside each tagged copy')
ax.legend(loc='lower center', frameon=False, ncol=2)
fig.tight_layout()

disjoint_fig = FIG_DIR / 'disjoint-union-component-tests.png'
save_matplotlib(fig, disjoint_fig)
plt.close(fig)

union_rows = [
    {
        'candidate': 'open slices in both components',
        'slice_A': '(0.2,0.82)',
        'slice_B': '(0.1,0.28) union (0.66,0.95)',
        'open_in_disjoint_union': True,
        'reason': 'each tagged slice is open in its own component',
    },
    {
        'candidate': 'endpoint included in A only',
        'slice_A': '[0,0.35)',
        'slice_B': '(0.4,0.6)',
        'open_in_disjoint_union': False,
        'reason': 'the A slice is not open in the interval component',
    },
    {
        'candidate': 'whole A plus open B slice',
        'slice_A': 'A',
        'slice_B': '(0.35,0.75)',
        'open_in_disjoint_union': True,
        'reason': 'whole components and empty slices are open in their components',
    },
]
disjoint_csv = TABLE_DIR / 'disjoint-union-open-tests.csv'
save_csv(union_rows, disjoint_csv)

display_artifact(disjoint_fig, width=820)
display(pd.DataFrame(union_rows))

## Quotients: Saturated Sets Are the Only Sets the Quotient Can See

A quotient topology starts with a surjective map `q:X -> Y`. A subset `V` of `Y` is declared open exactly when `q^{-1}(V)` is open in `X`. When `Y` is the set of equivalence classes of a relation on `X`, the quotient map sends each point to its class. The subsets of `X` that are preimages of subsets of `Y` are precisely the saturated subsets: once they contain a point, they contain its whole equivalence class.

This simple rule explains many standard constructions. Identifying the two endpoints of an interval produces a circle. Identifying opposite edges of a square produces a torus or projective-style variants depending on the arrow pattern. Identifying all nonzero scalar multiples in `R^{n+1} minus {0}` produces real projective space. Identifying all integer translates in `R` produces `R/Z`, homeomorphic to the circle through the exponential quotient map.

Quotient maps are powerful because they let us build spaces that are awkward to describe directly. They are dangerous because good properties do not automatically descend. Hausdorffness, first countability, and local Euclidean behavior need separate checks. The figure below focuses on what is visible before and after the quotient: which points are glued, which subsets are saturated, and which numerical residual confirms that integer translates have the same image on the circle.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

# Interval endpoints identified to a circle.
ax = axes[0]
ax.set_title('I / 0~1 gives a circle')
ax.plot([0, 1], [0, 0], color='0.35', lw=4)
ax.scatter([0, 1], [0, 0], s=90, color='#dc2626', zorder=3)
ax.text(0, -0.12, '0', ha='center')
ax.text(1, -0.12, '1', ha='center')
theta = np.linspace(0, 2 * np.pi, 220)
ax.plot(1.75 + 0.35 * np.cos(theta), 0.35 * np.sin(theta), color='#2563eb', lw=3)
ax.annotate('', xy=(1.35, 0), xytext=(1.07, 0), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.text(0.5, 0.16, 'endpoints identified', ha='center', color='#991b1b')
ax.set_aspect('equal')
ax.set_xlim(-0.15, 2.2)
ax.set_ylim(-0.55, 0.55)
ax.axis('off')

# Square edge identifications.
ax = axes[1]
ax.set_title('edge-pair quotient data')
ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color='0.2', lw=2)
ax.annotate('', xy=(0.78, -0.05), xytext=(0.22, -0.05), arrowprops=dict(arrowstyle='->', color='#2563eb', lw=2))
ax.annotate('', xy=(0.78, 1.05), xytext=(0.22, 1.05), arrowprops=dict(arrowstyle='->', color='#2563eb', lw=2))
ax.annotate('', xy=(-0.05, 0.78), xytext=(-0.05, 0.22), arrowprops=dict(arrowstyle='->', color='#f97316', lw=2))
ax.annotate('', xy=(1.05, 0.78), xytext=(1.05, 0.22), arrowprops=dict(arrowstyle='->', color='#f97316', lw=2))
ax.text(0.5, -0.18, 'a edges', ha='center', color='#1d4ed8')
ax.text(1.18, 0.5, 'b edges', va='center', color='#c2410c')
ax.set_aspect('equal')
ax.set_xlim(-0.22, 1.35)
ax.set_ylim(-0.3, 1.18)
ax.axis('off')

# R/Z saturated interval translates and circle image.
ax = axes[2]
ax.set_title('R/Z: saturated translates wrap to S1')
for k in range(-2, 4):
    ax.plot([k + 0.18, k + 0.42], [0, 0], color='#16a34a', lw=7, solid_capstyle='round')
    ax.text(k, -0.12, str(k), ha='center', fontsize=8)
ax.plot([-2.3, 3.3], [0, 0], color='0.6', lw=1)
cx, cy = 0.5, 0.68
ax.plot(cx + 0.25 * np.cos(theta), cy + 0.25 * np.sin(theta), color='#2563eb', lw=2.5)
arc = np.linspace(2 * np.pi * 0.18, 2 * np.pi * 0.42, 70)
ax.plot(cx + 0.25 * np.cos(arc), cy + 0.25 * np.sin(arc), color='#16a34a', lw=5)
ax.annotate('', xy=(0.5, 0.38), xytext=(0.5, 0.08), arrowprops=dict(arrowstyle='->', lw=1.4))
ax.set_xlim(-2.35, 3.35)
ax.set_ylim(-0.28, 1.05)
ax.axis('off')

fig.tight_layout()
quotient_fig = FIG_DIR / 'quotient-identification-gallery.png'
save_matplotlib(fig, quotient_fig)
plt.close(fig)

samples = np.linspace(-1.25, 1.25, 23)
shifts = np.arange(-3, 4)
residuals = []
for x in samples:
    base = np.exp(2j * np.pi * x)
    for k in shifts:
        residuals.append(abs(np.exp(2j * np.pi * (x + k)) - base))
quotient_checks = {
    'interval_endpoint_identification': {'q_0_equals_q_1': True},
    'square_edge_pairs_recorded': 2,
    'integer_translate_intervals_drawn': 6,
    'max_exp_integer_translate_residual': float(max(residuals)),
    'sampled_translate_residual_below_1e_minus_12': bool(max(residuals) < 1e-12),
}
quotient_json = CHECK_DIR / 'quotient-identification-checks.json'
save_json(quotient_checks, quotient_json)

display_artifact(quotient_fig, width=900)
display_artifact(quotient_json)

## Adjunction Spaces and Quotient Pathologies

An adjunction space is the standard way to attach one space to another. Start with a space `X`, a subspace `A subset X`, a map `f:A -> Y`, and the disjoint union `X + Y`. Then impose the equivalence relation that identifies each `a` in `A` with the point `f(a)` in `Y`. The result is written informally as `X cup_f Y`. This construction is the model behind attaching cells, cones, doubles of manifolds with boundary, and many later CW-complex constructions. Computationally, the important point is that adjunction is not a new primitive: it is disjoint union followed by a quotient.

Quotients also produce the main pathologies in the chapter. The problem is not that quotient maps are discontinuous; by definition the quotient topology makes the quotient map continuous. The problem is that saturated open neighborhoods in the original space may not be rich enough to separate different equivalence classes. For example, the orbit quotient of the general linear group action on `R^n` has two classes, the zero orbit and the nonzero orbit, but every saturated open neighborhood of the zero orbit contains the nonzero orbit too. This yields a two-point non-Hausdorff quotient. The line with two origins is locally Euclidean and second countable, yet the two origin classes cannot be separated by disjoint neighborhoods. These are warnings for manifold building: local coordinate charts do not replace Hausdorff checks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.1))

# Adjunction schematic.
ax = axes[0]
ax.set_title('adjunction = disjoint union then quotient')
# X disk-like blob and A boundary arc.
phi = np.linspace(0, 2 * np.pi, 200)
ax.plot(-0.9 + 0.45 * np.cos(phi), 0.1 + 0.35 * np.sin(phi), color='#2563eb', lw=2)
arc = np.linspace(-0.9, 0.9, 60)
ax.plot(-0.9 + 0.45 * np.cos(arc), 0.1 + 0.35 * np.sin(arc), color='#dc2626', lw=4)
ax.text(-0.9, -0.42, 'X with A highlighted', ha='center')
# Y target interval.
ax.plot([0.25, 1.15], [0.1, 0.1], color='#16a34a', lw=5, solid_capstyle='round')
ax.text(0.7, -0.42, 'Y', ha='center')
ax.annotate('f:A->Y', xy=(0.25, 0.18), xytext=(-0.25, 0.55), arrowprops=dict(arrowstyle='->'))
ax.annotate('quotient', xy=(1.55, 0.1), xytext=(1.23, 0.1), arrowprops=dict(arrowstyle='->'))
ax.plot(1.95 + 0.45 * np.cos(phi), 0.1 + 0.35 * np.sin(phi), color='#2563eb', lw=2)
ax.plot([1.5, 2.4], [0.1, 0.1], color='#16a34a', lw=5, solid_capstyle='round')
ax.text(1.95, -0.42, 'X cup_f Y', ha='center')
ax.set_aspect('equal')
ax.set_xlim(-1.55, 2.65)
ax.set_ylim(-0.62, 0.85)
ax.axis('off')

# Two-point orbit quotient topology.
ax = axes[1]
ax.set_title('orbit quotient: two points, not Hausdorff')
ax.scatter([0, 1], [0, 0], s=420, color=['#16a34a', '#dc2626'], edgecolors='0.2', zorder=3)
ax.text(0, -0.24, 'a = nonzero orbit', ha='center')
ax.text(1, -0.24, 'b = zero orbit', ha='center')
ax.add_patch(plt.Circle((0, 0), 0.32, fill=False, color='#16a34a', lw=2))
ax.add_patch(plt.Circle((0.5, 0), 0.78, fill=False, color='#7c3aed', lw=2, linestyle='--'))
ax.text(0, 0.43, 'open {a}', ha='center', color='#166534')
ax.text(0.5, 0.83, 'only open set containing b is {a,b}', ha='center', color='#5b21b6')
ax.set_xlim(-0.62, 1.62)
ax.set_ylim(-0.55, 1.05)
ax.axis('off')

# Line with two origins schematic.
ax = axes[2]
ax.set_title('line with two origins: local lines, global failure')
x = np.linspace(-1.5, 1.5, 200)
ax.plot(x[x < -0.08], np.zeros_like(x[x < -0.08]), color='0.45', lw=3)
ax.plot(x[x > 0.08], np.zeros_like(x[x > 0.08]), color='0.45', lw=3)
ax.scatter([0, 0], [0.18, -0.18], s=90, color=['#2563eb', '#f97316'], zorder=4)
ax.plot([-0.55, -0.08], [0.18, 0.02], color='#2563eb', lw=2)
ax.plot([0.08, 0.55], [0.02, 0.18], color='#2563eb', lw=2)
ax.plot([-0.55, -0.08], [-0.18, -0.02], color='#f97316', lw=2)
ax.plot([0.08, 0.55], [-0.02, -0.18], color='#f97316', lw=2)
ax.text(0, 0.34, 'origin 0+', ha='center', color='#1d4ed8')
ax.text(0, -0.36, 'origin 0-', ha='center', color='#c2410c')
ax.text(0.02, 0.05, 'punctured neighborhoods overlap', ha='left', fontsize=8)
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-0.62, 0.62)
ax.axis('off')

fig.tight_layout()
adjunction_fig = FIG_DIR / 'adjunction-and-quotient-pathologies.png'
save_matplotlib(fig, adjunction_fig)
plt.close(fig)

pathology_checks = {
    'adjunction_modeled_as': 'disjoint union followed by quotient identifying a with f(a)',
    'orbit_quotient_open_sets': ['empty', '{nonzero}', '{nonzero, zero}'],
    'orbit_quotient_is_T1': False,
    'orbit_quotient_is_Hausdorff': False,
    'line_with_two_origins_locally_euclidean': True,
    'line_with_two_origins_Hausdorff': False,
    'reason_for_failure': 'the two origin classes have no disjoint saturated neighborhoods',
}
pathology_json = CHECK_DIR / 'adjunction-pathology-checks.json'
save_json(pathology_checks, pathology_json)

display_artifact(adjunction_fig, width=900)
display_artifact(pathology_json)

## Applied Lab: Audit a Finite Quotient Before Trusting It

The following lab turns the quotient-topology definition into a small calculator. A finite topology is represented by its set of open subsets. A quotient is represented by blocks of points, one block per equivalence class. A subset of quotient classes is open exactly when its preimage, the union of the corresponding blocks, is open in the source topology.

This finite model is intentionally small, but it mirrors the real workflow for larger quotient spaces. First identify the equivalence classes. Then test candidate opens by pulling them back to the original space. Then test separation properties on the quotient, not on the original space alone. A Hausdorff source can still produce a non-Hausdorff quotient if the quotient map is not open or if orbit closures collide; an open quotient map has better behavior, but even then the saturated-subset check is the essential mechanism.

Try changing the blocks or source opens in this cell. The table updates the quotient open sets, the `T1` result, and the Hausdorff result. The examples are chosen to separate three behaviors: a quotient of a finite discrete source that remains discrete, the Sierpinski two-point topology as a minimal non-Hausdorff quotient model, and the orbit-style two-point topology where the nonzero orbit is open but the zero orbit cannot be isolated.

In [ ]:
def powerset(items):
    items = list(items)
    for mask in range(1 << len(items)):
        yield frozenset(items[i] for i in range(len(items)) if mask & (1 << i))

def quotient_topology(source_points, source_opens, blocks):
    source_opens = {frozenset(s) for s in source_opens}
    block_items = {name: frozenset(points) for name, points in blocks.items()}
    quotient_points = tuple(block_items)
    quotient_opens = set()
    for candidate in powerset(quotient_points):
        preimage = frozenset().union(*(block_items[name] for name in candidate)) if candidate else frozenset()
        if preimage in source_opens:
            quotient_opens.add(frozenset(candidate))
    return quotient_points, quotient_opens

def is_T1(points, opens):
    points = list(points)
    opens = set(opens)
    for p in points:
        complement = frozenset(q for q in points if q != p)
        if complement not in opens:
            return False
    return True

def is_hausdorff(points, opens):
    points = list(points)
    opens = set(opens)
    for i, p in enumerate(points):
        for q in points[i + 1:]:
            separated = False
            for U in opens:
                if p not in U:
                    continue
                for V in opens:
                    if q in V and U.isdisjoint(V):
                        separated = True
                        break
                if separated:
                    break
            if not separated:
                return False
    return True

def format_opens(opens):
    if not opens:
        return 'none'
    formatted = []
    for item in sorted(opens, key=lambda s: (len(s), sorted(s))):
        formatted.append('{' + ','.join(sorted(item)) + '}')
    return '; '.join(formatted)

examples = [
    {
        'name': 'finite discrete quotient',
        'source_points': ['0', '1', '2', '3'],
        'source_opens': list(powerset(['0', '1', '2', '3'])),
        'blocks': {'a': ['0', '2'], 'b': ['1'], 'c': ['3']},
        'expected_hausdorff': True,
    },
    {
        'name': 'Sierpinski model',
        'source_points': ['x', 'y'],
        'source_opens': [[], ['x'], ['x', 'y']],
        'blocks': {'xbar': ['x'], 'ybar': ['y']},
        'expected_hausdorff': False,
    },
    {
        'name': 'orbit-style two point quotient',
        'source_points': ['nonzero', 'zero'],
        'source_opens': [[], ['nonzero'], ['nonzero', 'zero']],
        'blocks': {'a_nonzero': ['nonzero'], 'b_zero': ['zero']},
        'expected_hausdorff': False,
    },
]

lab_rows = []
for example in examples:
    q_points, q_opens = quotient_topology(example['source_points'], example['source_opens'], example['blocks'])
    row = {
        'example': example['name'],
        'quotient_points': ','.join(q_points),
        'quotient_open_sets': format_opens(q_opens),
        'open_set_count': len(q_opens),
        'T1': is_T1(q_points, q_opens),
        'Hausdorff': is_hausdorff(q_points, q_opens),
        'expected_Hausdorff': example['expected_hausdorff'],
    }
    lab_rows.append(row)

lab_csv = TABLE_DIR / 'finite-quotient-lab-results.csv'
save_csv(lab_rows, lab_csv)
lab_json = CHECK_DIR / 'finite-quotient-lab-checks.json'
save_json({'rows': lab_rows, 'all_expectations_met': all(r['Hausdorff'] == r['expected_Hausdorff'] for r in lab_rows)}, lab_json)

display(pd.DataFrame(lab_rows))
display_artifact(lab_json)

## Takeaways

The chapter has one governing pattern: a topology on a constructed set is best understood by its canonical maps. Subspaces are controlled by inclusion into an ambient space. Products are controlled by projections. Disjoint unions are controlled by injections from the components. Quotients are controlled by a surjective map from the old space, and the only old-space subsets that matter are saturated subsets.

The constructions are not merely formal. They build core manifolds and near-manifolds: graphs and spheres as subspaces, tori as products and quotients, circles as endpoint or integer-translate quotients, cones and doubles as adjunction spaces, and projective spaces as orbit spaces. They also create the examples that force caution. A quotient can be locally Euclidean and second countable while failing Hausdorffness. A continuous injective map can fail to be an embedding if the inverse image topology is wrong. Infinite products can behave differently depending on whether product or box neighborhoods are allowed.

A practical audit for any new space from this chapter is therefore short but strict. Name the canonical map. State the open-set test. Check the characteristic property needed for the maps you want to use. Record which properties are inherited and which require proof. For quotient constructions, test saturation and separation before calling the result a manifold.

In [ ]:
# final_sanity: executable checks for core identities, artifact integrity, and notebook structure.
expected_artifacts = [
    CHECK_DIR / 'visual-storyboard.json',
    FIG_DIR / 'relative-openness-subspace.png',
    CHECK_DIR / 'subspace-relative-openness-checks.json',
    FIG_DIR / 'embedding-vs-wrapping.png',
    CHECK_DIR / 'embedding-vs-wrapping-checks.json',
    FIG_DIR / 'characteristic-property-gates.png',
    CHECK_DIR / 'characteristic-property-gates-checks.json',
    HTML_DIR / 'product-vs-box-diagonal.html',
    CHECK_DIR / 'product-vs-box-diagonal-checks.json',
    FIG_DIR / 'disjoint-union-component-tests.png',
    TABLE_DIR / 'disjoint-union-open-tests.csv',
    FIG_DIR / 'quotient-identification-gallery.png',
    CHECK_DIR / 'quotient-identification-checks.json',
    FIG_DIR / 'adjunction-and-quotient-pathologies.png',
    CHECK_DIR / 'adjunction-pathology-checks.json',
    TABLE_DIR / 'finite-quotient-lab-results.csv',
    CHECK_DIR / 'finite-quotient-lab-checks.json',
]
assert_artifacts(expected_artifacts, min_bytes=64)

with (CHECK_DIR / 'visual-storyboard.json').open(encoding='utf-8') as handle:
    storyboard_data = json.load(handle)
assert storyboard_data['source_span'] == 'printed pp. 49-84, PDF pp. 67-102'
assert len(storyboard_data['items']) >= 7
assert all(item.get('validation') and item.get('inspection_target') for item in storyboard_data['items'])

with (CHECK_DIR / 'subspace-relative-openness-checks.json').open(encoding='utf-8') as handle:
    subspace_data = json.load(handle)
assert subspace_data['all_sampled_singletons_open_in_subspace']
assert subspace_data['zero_is_ambient_limit_point'] and not subspace_data['zero_is_in_S2']

with (CHECK_DIR / 'embedding-vs-wrapping-checks.json').open(encoding='utf-8') as handle:
    embedding_data = json.load(handle)
assert embedding_data['parabola_embedding_detected']
assert not embedding_data['wrapping_map_embedding_detected']
assert embedding_data['wrap_image_distance_for_s_0_and_1_minus_eps'] < 0.01
assert embedding_data['wrap_parameter_jump_for_same_pair'] > 0.99

with (CHECK_DIR / 'characteristic-property-gates-checks.json').open(encoding='utf-8') as handle:
    gates_data = json.load(handle)
assert gates_data['is_directed_acyclic_graph']
assert set(gates_data['construction_groups']) == {'Adjunction', 'Disjoint union', 'Product', 'Quotient', 'Subspace'}

with (CHECK_DIR / 'product-vs-box-diagonal-checks.json').open(encoding='utf-8') as handle:
    product_data = json.load(handle)
assert product_data['diagonal_map_product_topology_continuous']
assert not product_data['diagonal_map_box_topology_continuous']
assert product_data['infinite_box_preimage_is_singleton_zero']

with (CHECK_DIR / 'quotient-identification-checks.json').open(encoding='utf-8') as handle:
    quotient_data = json.load(handle)
assert quotient_data['sampled_translate_residual_below_1e_minus_12']

with (CHECK_DIR / 'adjunction-pathology-checks.json').open(encoding='utf-8') as handle:
    pathology_data = json.load(handle)
assert not pathology_data['orbit_quotient_is_Hausdorff']
assert not pathology_data['line_with_two_origins_Hausdorff']

with (CHECK_DIR / 'finite-quotient-lab-checks.json').open(encoding='utf-8') as handle:
    lab_data = json.load(handle)
assert lab_data['all_expectations_met']

png_stats = [image_stats(path) for path in expected_artifacts if path.suffix.lower() == '.png']
assert png_stats and all(item['max_channel_stddev'] > 1.0 for item in png_stats)

notebook_path = BOOK_ROOT / 'chapter-03-new-spaces-from-old' / '03-new-spaces-from-old.ipynb'
if notebook_path.exists():
    notebook_data = json.loads(notebook_path.read_text(encoding='utf-8'))
    markdown_cells = [cell for cell in notebook_data['cells'] if cell['cell_type'] == 'markdown']
    code_cells = [cell for cell in notebook_data['cells'] if cell['cell_type'] == 'code']
    def cell_text(cell):
        source = cell.get('source', '')
        return ''.join(source) if isinstance(source, list) else source
    markdown_word_count = sum(len(cell_text(cell).split()) for cell in markdown_cells)
    assert markdown_word_count >= 1200
    assert len(code_cells) >= 5
else:
    markdown_word_count = None

final_sanity = {
    'artifact_count_checked': len(expected_artifacts),
    'png_count_checked': len(png_stats),
    'markdown_word_count': markdown_word_count,
    'core_checks': [
        'subspace singleton isolation and ambient closure warning',
        'embedding versus continuous injective wrapping inverse check',
        'characteristic-property graph acyclic and complete by construction group',
        'product topology versus box topology diagonal test',
        'integer-translate quotient residual for R/Z -> S1',
        'finite quotient Hausdorff and T1 audit expectations',
        'PNG artifacts nonblank by channel standard deviation',
    ],
}
final_sanity_path = CHECK_DIR / 'final-sanity-summary.json'
save_json(final_sanity, final_sanity_path)
display(pd.DataFrame(png_stats))
display_artifact(final_sanity_path)